In [ ]:
import random
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.callbacks import EarlyStopping


# ============================================================
# 1. Reproducerbarhet
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


# ============================================================
# 2. Läs in California Housing
# ============================================================
# Datasetet har 20 640 observationer och åtta numeriska variabler.
#
# Vi använder endast inputvariablerna, inte MedHouseVal, eftersom
# targetvariabeln normalt inte ska användas för att imputera input.
#
# Datasetet saknar ursprungliga missing values. Det är praktiskt
# vid utvärdering eftersom vi kan skapa missing values själva och
# alltid har ett känt facit.
# ============================================================

housing = fetch_california_housing(as_frame=True)

df = housing.data.copy()

print("Datasetets dimension:", df.shape)
print("\nKolumner:")
print(df.columns.tolist())

print("\nAntal ursprungliga missing values:")
print(df.isna().sum())

assert df.isna().sum().sum() == 0, (
    "Datasetet förväntades vara komplett."
)


# ============================================================
# 3. Dela data i train, validation och test
# ============================================================
# Train används för träning.
# Validation används för early stopping.
# Test används endast för slutlig utvärdering.
# ============================================================

df_train, df_temp = train_test_split(
    df,
    test_size=0.30,
    random_state=SEED
)

df_val, df_test_original = train_test_split(
    df_temp,
    test_size=0.50,
    random_state=SEED
)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test_original = df_test_original.reset_index(drop=True)

print("\nAntal rader:")
print("Train:", len(df_train))
print("Validation:", len(df_val))
print("Test:", len(df_test_original))


# ============================================================
# 4. Beräkna medianer från träningsdata
# ============================================================
# Medianerna används som tekniska ersättningsvärden i modellens
# input. Modellen kan inte ta emot NaN direkt.
#
# Medianerna beräknas endast från träningsdata.
# ============================================================

train_medians = df_train.median()


# ============================================================
# 5. Standardisera data
# ============================================================
# Skalaren anpassas endast på träningsdata.
#
# Validation och test använder samma skalning genom transform(),
# aldrig fit_transform().
# ============================================================

scaler = StandardScaler()

train_scaled = scaler.fit_transform(df_train)
val_scaled = scaler.transform(df_val)
test_original_scaled = scaler.transform(df_test_original)

train_scaled = train_scaled.astype(np.float32)
val_scaled = val_scaled.astype(np.float32)
test_original_scaled = test_original_scaled.astype(np.float32)

n_features = train_scaled.shape[1]
feature_names = df_train.columns.tolist()


# Medianerna måste också uttryckas i den standardiserade skalan,
# eftersom de ska placeras i den korrumperade modellinputen.
median_scaled = scaler.transform(
    pd.DataFrame(
        [train_medians],
        columns=feature_names
    )
)[0].astype(np.float32)


# ============================================================
# 6. Funktion för att skapa en slumpmässig missing-mask
# ============================================================
# True betyder att värdet ska döljas.
#
# Vi ser till att:
# - varje rad får minst ett dolt värde,
# - varje rad behåller minst ett observerat värde.
#
# Modellen måste ha någon information kvar för att kunna göra en
# rekonstruktion.
# ============================================================

def create_missing_mask(
    number_of_rows,
    number_of_features,
    missing_fraction,
    rng
):
    mask = (
        rng.random(
            (number_of_rows, number_of_features)
        ) < missing_fraction
    )

    for row_index in range(number_of_rows):
        number_missing = mask[row_index].sum()

        # Om inget värde maskerades, maskera en slumpmässig kolumn.
        if number_missing == 0:
            column_index = rng.integers(number_of_features)
            mask[row_index, column_index] = True

        # Om alla värden maskerades, gör en kolumn observerad igen.
        elif number_missing == number_of_features:
            column_index = rng.integers(number_of_features)
            mask[row_index, column_index] = False

    return mask


# ============================================================
# 7. Funktion för att korrumpera data
# ============================================================
# De utvalda värdena ersätts med träningsmedianen.
#
# Input till modellen består av:
#
#   [korrumperade värden, missing-mask]
#
# Masken gör att modellen kan skilja mellan:
#
# - ett verkligt värde nära medianen,
# - ett värde som ersatts med medianen eftersom det saknas.
# ============================================================

def corrupt_data(clean_scaled, missing_mask):
    corrupted_scaled = clean_scaled.copy()

    for column_index in range(n_features):
        corrupted_scaled[
            missing_mask[:, column_index],
            column_index
        ] = median_scaled[column_index]

    model_input = np.concatenate(
        [
            corrupted_scaled,
            missing_mask.astype(np.float32)
        ],
        axis=1
    )

    return model_input.astype(np.float32)


# ============================================================
# 8. Targets med information om vilka värden som doldes
# ============================================================
# y_true innehåller:
#
#   [originalvärden, loss-mask]
#
# Originalvärdena används som facit.
# Loss-masken anger vilka celler som var artificiellt dolda.
#
# Loss beräknas huvudsakligen på dolda värden, eftersom det är
# imputering som modellen ska lära sig.
# ============================================================

def create_targets(clean_scaled, missing_mask):
    return np.concatenate(
        [
            clean_scaled,
            missing_mask.astype(np.float32)
        ],
        axis=1
    ).astype(np.float32)


# ============================================================
# 9. Skapa flera korrumperade träningskopior
# ============================================================
# Varje träningsrad används flera gånger med olika missing-masker.
#
# Det ger modellen fler typer av imputationsproblem utan att ändra
# originaldatan.
# ============================================================

def create_denoising_dataset(
    clean_scaled,
    number_of_copies,
    missing_fraction,
    random_state
):
    rng = np.random.default_rng(random_state)

    input_copies = []
    target_copies = []

    for _ in range(number_of_copies):
        missing_mask = create_missing_mask(
            number_of_rows=len(clean_scaled),
            number_of_features=n_features,
            missing_fraction=missing_fraction,
            rng=rng
        )

        model_input = corrupt_data(
            clean_scaled,
            missing_mask
        )

        model_target = create_targets(
            clean_scaled,
            missing_mask
        )

        input_copies.append(model_input)
        target_copies.append(model_target)

    X = np.concatenate(input_copies, axis=0)
    y = np.concatenate(target_copies, axis=0)

    # Blanda de skapade kopiorna.
    permutation = rng.permutation(len(X))

    return X[permutation], y[permutation]


TRAIN_MISSING_FRACTION = 0.20

X_train, y_train = create_denoising_dataset(
    clean_scaled=train_scaled,
    number_of_copies=4,
    missing_fraction=TRAIN_MISSING_FRACTION,
    random_state=SEED
)

X_val, y_val = create_denoising_dataset(
    clean_scaled=val_scaled,
    number_of_copies=1,
    missing_fraction=TRAIN_MISSING_FRACTION,
    random_state=SEED + 1
)

print("\nDenoising-data:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_val:", X_val.shape)
print("y_val:", y_val.shape)


# ============================================================
# 10. Maskerad loss
# ============================================================
# Loss beräknas på de artificiellt dolda värdena.
#
# Ett litet lossbidrag läggs även på observerade värden. Det hjälper
# autoencodern att lära sig datastrukturen och stabiliserar träningen.
#
# Dolda värden får full vikt:       1.0
# Observerade värden får liten vikt: 0.05
# ============================================================

OBSERVED_VALUE_WEIGHT = 0.05


def masked_mse(y_true_with_mask, y_pred):
    y_true = y_true_with_mask[:, :n_features]
    missing_mask = y_true_with_mask[:, n_features:]

    weights = (
        missing_mask
        + OBSERVED_VALUE_WEIGHT * (1.0 - missing_mask)
    )

    squared_errors = tf.square(y_true - y_pred)
    weighted_squared_errors = weights * squared_errors

    return (
        tf.reduce_sum(weighted_squared_errors)
        / (tf.reduce_sum(weights) + 1e-8)
    )


# ============================================================
# 11. Bygg autoencodern
# ============================================================
# Input har 16 dimensioner:
#
# - 8 korrumperade datavärden,
# - 8 maskvärden.
#
# Output har endast de åtta rekonstruerade datavärdena.
# Missing-masken ska användas som information men ska inte själv
# rekonstrueras.
# ============================================================

input_dim = X_train.shape[1]

input_layer = Input(
    shape=(input_dim,),
    name="corrupted_values_and_mask"
)

encoder = Dense(
    64,
    activation="relu",
    name="encoder_1"
)(input_layer)

encoder = Dense(
    32,
    activation="relu",
    name="encoder_2"
)(encoder)

# Bottleneck: en komprimerad representation av observationen.
bottleneck = Dense(
    12,
    activation="relu",
    name="bottleneck"
)(encoder)

decoder = Dense(
    32,
    activation="relu",
    name="decoder_1"
)(bottleneck)

decoder = Dense(
    64,
    activation="relu",
    name="decoder_2"
)(decoder)

# Linjär output eftersom samtliga variabler är numeriska.
output_layer = Dense(
    n_features,
    activation="linear",
    name="reconstructed_values"
)(decoder)

autoencoder = Model(
    inputs=input_layer,
    outputs=output_layer
)

autoencoder.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss=masked_mse
)

autoencoder.summary()


# ============================================================
# 12. Träna modellen
# ============================================================

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=15,
    restore_best_weights=True,
    verbose=1
)

history = autoencoder.fit(
    X_train,
    y_train,
    epochs=200,
    batch_size=128,
    validation_data=(X_val, y_val),
    callbacks=[early_stopping],
    verbose=1
)


# ============================================================
# 13. Skapa artificiella missing values i testdata
# ============================================================
# Samma testmask används för:
#
# - autoencodern,
# - medianimputeringen,
# - samtliga jämförelser.
#
# Därmed jämförs metoderna på exakt samma värden.
# ============================================================

TEST_MISSING_FRACTION = 0.20

test_rng = np.random.default_rng(SEED + 2)

test_missing_mask = create_missing_mask(
    number_of_rows=len(test_original_scaled),
    number_of_features=n_features,
    missing_fraction=TEST_MISSING_FRACTION,
    rng=test_rng
)

X_test = corrupt_data(
    test_original_scaled,
    test_missing_mask
)


# En DataFrame som motsvarar testdata med riktiga NaN-värden.
df_test_masked = df_test_original.copy()

for column_index, column_name in enumerate(feature_names):
    df_test_masked.loc[
        test_missing_mask[:, column_index],
        column_name
    ] = np.nan


print("\nAntal artificiellt saknade testvärden per variabel:")
print(df_test_masked.isna().sum())


# ============================================================
# 14. Rekonstruera testdata
# ============================================================

reconstructed_test_scaled = autoencoder.predict(
    X_test,
    verbose=0
)

reconstructed_test = scaler.inverse_transform(
    reconstructed_test_scaled
)

reconstructed_df = pd.DataFrame(
    reconstructed_test,
    columns=feature_names
)


# ============================================================
# 15. Skapa en fullständigt imputerad DataFrame
# ============================================================
# Endast saknade celler ersätts med modellens rekonstruktion.
# Observerade originalvärden lämnas oförändrade.
# ============================================================

df_test_imputed = df_test_masked.copy()

for column_index, column_name in enumerate(feature_names):
    rows_missing = test_missing_mask[:, column_index]

    df_test_imputed.loc[
        rows_missing,
        column_name
    ] = reconstructed_df.loc[
        rows_missing,
        column_name
    ]


assert df_test_imputed.isna().sum().sum() == 0, (
    "Det finns fortfarande missing values efter imputering."
)


# ============================================================
# 16. Utvärdera varje variabel
# ============================================================
# Eftersom detta är regression använder vi:
#
# MAE:
#   Genomsnittligt absolut fel i variabelns ursprungliga enhet.
#
# RMSE:
#   Som MAE, men straffar stora fel hårdare.
#
# R²:
#   1.0 är perfekt.
#   0.0 motsvarar ungefär en konstant medelvärdesprediktion.
#   Negativt innebär att modellen är sämre än en sådan baseline.
#
# Median MAE:
#   MAE om alla missing values ersätts med träningsmedianen.
#
# Förbättring:
#   Procentuell förbättring av MAE jämfört med medianen.
#
# Inom 10/25 % av std:
#   Andel skattningar vars absoluta fel ligger inom en viss andel
#   av variabelns standardavvikelse i träningsdata.
# ============================================================

evaluation_rows = []
all_estimates = []

train_standard_deviations = df_train.std()

for column_index, column_name in enumerate(feature_names):
    rows_missing = test_missing_mask[:, column_index]
    test_row_indices = np.where(rows_missing)[0]

    actual_values = df_test_original.loc[
        rows_missing,
        column_name
    ].to_numpy()

    predicted_values = reconstructed_df.loc[
        rows_missing,
        column_name
    ].to_numpy()

    median_value = train_medians[column_name]

    median_predictions = np.full(
        shape=len(actual_values),
        fill_value=median_value
    )

    absolute_errors = np.abs(
        actual_values - predicted_values
    )

    median_absolute_errors = np.abs(
        actual_values - median_predictions
    )

    model_mae = mean_absolute_error(
        actual_values,
        predicted_values
    )

    model_rmse = np.sqrt(
        mean_squared_error(
            actual_values,
            predicted_values
        )
    )

    model_r2 = r2_score(
        actual_values,
        predicted_values
    )

    median_mae = mean_absolute_error(
        actual_values,
        median_predictions
    )

    # Positivt värde betyder att autoencodern är bättre.
    # Negativt värde betyder att medianen är bättre.
    improvement_percent = (
        (median_mae - model_mae)
        / median_mae
        * 100
    )

    feature_std = train_standard_deviations[column_name]

    tolerance_10 = 0.10 * feature_std
    tolerance_25 = 0.25 * feature_std

    within_10_percent_std = np.mean(
        absolute_errors <= tolerance_10
    )

    within_25_percent_std = np.mean(
        absolute_errors <= tolerance_25
    )

    evaluation_rows.append({
        "Feature": column_name,
        "NumberOfEstimates": len(actual_values),
        "Autoencoder_MAE": model_mae,
        "Autoencoder_RMSE": model_rmse,
        "Autoencoder_R2": model_r2,
        "Median_MAE": median_mae,
        "MAE_Improvement_Percent": improvement_percent,
        "Within_10pct_of_STD": within_10_percent_std,
        "Within_25pct_of_STD": within_25_percent_std
    })

    # Spara varje enskild skattning för senare inspektion.
    for (
        row_index,
        actual_value,
        predicted_value,
        median_prediction,
        absolute_error,
        median_absolute_error
    ) in zip(
        test_row_indices,
        actual_values,
        predicted_values,
        median_predictions,
        absolute_errors,
        median_absolute_errors
    ):
        all_estimates.append({
            "TestRow": int(row_index),
            "Feature": column_name,
            "OriginalValue": actual_value,
            "AutoencoderEstimate": predicted_value,
            "MedianEstimate": median_prediction,
            "AutoencoderAbsoluteError": absolute_error,
            "MedianAbsoluteError": median_absolute_error,
            "AutoencoderBetterThanMedian": (
                absolute_error < median_absolute_error
            )
        })


evaluation_df = pd.DataFrame(evaluation_rows)

all_estimates_df = pd.DataFrame(all_estimates)


# ============================================================
# 17. Sammanfattning per variabel
# ============================================================

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

print("\n")
print("=" * 120)
print("RESULTAT PER VARIABEL")
print("=" * 120)

print(
    evaluation_df.round({
        "Autoencoder_MAE": 4,
        "Autoencoder_RMSE": 4,
        "Autoencoder_R2": 4,
        "Median_MAE": 4,
        "MAE_Improvement_Percent": 2,
        "Within_10pct_of_STD": 4,
        "Within_25pct_of_STD": 4
    }).to_string(index=False)
)


# ============================================================
# 18. Total jämförelse över alla skattningar
# ============================================================
# Variablerna har olika skalor. Ett totalt MAE i originalskala är
# därför inte särskilt meningsfullt.
#
# Här visar vi i stället:
#
# - hur ofta autoencodern slår medianen,
# - totalt antal skattningar,
# - genomsnittlig förbättring per feature.
# ============================================================

total_number_of_estimates = len(all_estimates_df)

autoencoder_better_fraction = (
    all_estimates_df[
        "AutoencoderBetterThanMedian"
    ].mean()
)

mean_feature_improvement = evaluation_df[
    "MAE_Improvement_Percent"
].mean()

print("\n")
print("=" * 120)
print("TOTAL SAMMANFATTNING")
print("=" * 120)

print(
    f"Totalt antal skattade missing values: "
    f"{total_number_of_estimates}"
)

print(
    f"Autoencodern hade lägre absolut fel än medianen för: "
    f"{autoencoder_better_fraction:.2%} av alla skattningar"
)

print(
    f"Genomsnittlig MAE-förbättring per variabel: "
    f"{mean_feature_improvement:.2f}%"
)


# ============================================================
# 19. Visa enskilda skattningar
# ============================================================
# Tabellen innehåller:
#
# - vilken test-rad som imputerades,
# - vilken variabel,
# - det riktiga värdet,
# - autoencoderns skattning,
# - medianens skattning,
# - respektive absolut fel.
# ============================================================

print("\n")
print("=" * 120)
print("EXEMPEL PÅ ENSKILDA SKATTNINGAR")
print("=" * 120)

print(
    all_estimates_df
    .head(30)
    .round(4)
    .to_string(index=False)
)


# ============================================================
# 20. Visa de bästa och sämsta skattningarna
# ============================================================

print("\n")
print("=" * 120)
print("10 SKATTNINGAR MED LÄGST ABSOLUT FEL")
print("=" * 120)

print(
    all_estimates_df
    .sort_values("AutoencoderAbsoluteError")
    .head(10)
    .round(4)
    .to_string(index=False)
)


print("\n")
print("=" * 120)
print("10 SKATTNINGAR MED HÖGST ABSOLUT FEL")
print("=" * 120)

print(
    all_estimates_df
    .sort_values(
        "AutoencoderAbsoluteError",
        ascending=False
    )
    .head(10)
    .round(4)
    .to_string(index=False)
)


# ============================================================
# 21. Spara samtliga resultat
# ============================================================
# Här kan du kontrollera varje enskild skattning i exempelvis
# Excel, Python eller ett BI-verktyg.
# ============================================================

evaluation_df.to_csv(
    "imputation_accuracy_by_feature.csv",
    index=False
)

all_estimates_df.to_csv(
    "all_imputation_estimates.csv",
    index=False
)

df_test_original.to_csv(
    "test_original.csv",
    index=False
)

df_test_masked.to_csv(
    "test_with_missing_values.csv",
    index=False
)

df_test_imputed.to_csv(
    "test_imputed_by_autoencoder.csv",
    index=False
)

print("\nResultaten har sparats i:")
print("  imputation_accuracy_by_feature.csv")
print("  all_imputation_estimates.csv")
print("  test_original.csv")
print("  test_with_missing_values.csv")
print("  test_imputed_by_autoencoder.csv")

Datasetets dimension: (20640, 8)

Kolumner:
['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']

Antal ursprungliga missing values:
MedInc        0
HouseAge      0
AveRooms      0
AveBedrms     0
Population    0
AveOccup      0
Latitude      0
Longitude     0
dtype: int64

Antal rader:
Train: 14448
Validation: 3096
Test: 3096

Denoising-data:
X_train: (57792, 16)
y_train: (57792, 16)
X_val: (3096, 16)
y_val: (3096, 16)


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ corrupted_values_and_mask       │ (None, 16)             │             0 │
│ (InputLayer)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder_1 (Dense)               │ (None, 64)             │         1,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder_2 (Dense)               │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bottleneck (Dense)              │ (None, 12)             │           396 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_1 (Dense)               │ (None, 32)             │           416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_2 (Dense)               │ (None, 64)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reconstructed_values (Dense)    │ (None, 8)              │           520 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,612 (25.83 KB)

 Trainable params: 6,612 (25.83 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/200
452/452 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - loss: 0.5688 - val_loss: 0.3753
Epoch 2/200
452/452 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.4238 - val_loss: 0.3494
Epoch 3/200
452/452 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.4041 - val_loss: 0.3417
Epoch 4/200
452/452 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.3844 - val_loss: 0.3363
Epoch 5/200
452/452 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.3907 - val_loss: 0.3321
Epoch 6/200
452/452 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - loss: 0.3843 - val_loss: 0.3317
Epoch 7/200
452/452 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.3730 - val_loss: 0.3299
Epoch 8/200
452/452 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.3660 - val_loss: 0.3267
Epoch 9/200
452/452 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.3611 - val_loss: 0.3255
Epoch 10/200
452/452 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.3589 - val_loss: 0.3249
Epoch 11/200
452/452 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - loss: 0.3569 - val_loss: 0.3240
Epoch 12/200
452/452 ━━━━━━━━━━━━━━━━━━━

In [4]:
# ============================================================
# Utvärdera modellens rekonstruktion
# ============================================================
# Endast artificiellt dolda värden utvärderas.
#
# Observerade värden tas alltså inte med i resultaten eftersom
# modellen redan fick se dessa värden i sin input.
# ============================================================

evaluation_rows = []
all_estimates = []

train_standard_deviations = df_train.std()

for column_index, column_name in enumerate(feature_names):

    # Vilka testvärden i den aktuella kolumnen doldes?
    rows_missing = test_missing_mask[:, column_index]
    test_row_indices = np.where(rows_missing)[0]

    # Originalvärdena är vårt facit.
    actual_values = df_test_original.loc[
        rows_missing,
        column_name
    ].to_numpy()

    # Autoencoderns rekonstruktioner.
    predicted_values = reconstructed_df.loc[
        rows_missing,
        column_name
    ].to_numpy()

    # Medianimputering används som baseline.
    median_value = train_medians[column_name]

    median_predictions = np.full(
        len(actual_values),
        median_value
    )

    # --------------------------------------------------------
    # Fel för varje enskild skattning
    # --------------------------------------------------------

    absolute_errors = np.abs(
        actual_values - predicted_values
    )

    median_absolute_errors = np.abs(
        actual_values - median_predictions
    )

    # --------------------------------------------------------
    # Sammanfattande mått för variabeln
    # --------------------------------------------------------

    model_mae = mean_absolute_error(
        actual_values,
        predicted_values
    )

    model_rmse = np.sqrt(
        mean_squared_error(
            actual_values,
            predicted_values
        )
    )

    model_r2 = r2_score(
        actual_values,
        predicted_values
    )

    median_mae = mean_absolute_error(
        actual_values,
        median_predictions
    )

    # Positivt värde innebär att autoencodern är bättre.
    # Negativt värde innebär att medianen är bättre.
    improvement_percent = (
        (median_mae - model_mae)
        / (median_mae + 1e-8)
        * 100
    )

    # Hur ofta autoencodern ger lägre absolut fel än medianen.
    better_than_median = (
        absolute_errors < median_absolute_errors
    )

    equal_to_median = np.isclose(
        absolute_errors,
        median_absolute_errors
    )

    # Eftersom vanliga accuracy-mått inte fungerar för kontinuerliga
    # värden definieras träff som att skattningen ligger inom en
    # viss tolerans från facit.
    feature_std = train_standard_deviations[column_name]

    tolerance_10 = 0.10 * feature_std
    tolerance_25 = 0.25 * feature_std

    within_10_percent_std = np.mean(
        absolute_errors <= tolerance_10
    )

    within_25_percent_std = np.mean(
        absolute_errors <= tolerance_25
    )

    evaluation_rows.append({
        "Variabel": column_name,
        "Antal skattningar": len(actual_values),
        "Autoencoder MAE": model_mae,
        "Median MAE": median_mae,
        "Förbättring mot median (%)": improvement_percent,
        "RMSE": model_rmse,
        "R2": model_r2,
        "Bättre än median (%)": (
            better_than_median.mean() * 100
        ),
        "Inom 10 % av std (%)": (
            within_10_percent_std * 100
        ),
        "Inom 25 % av std (%)": (
            within_25_percent_std * 100
        )
    })

    # Spara resultaten i minnet för utskrift.
    # Inga CSV-filer skapas.
    for (
        row_index,
        actual_value,
        predicted_value,
        median_prediction,
        absolute_error,
        median_absolute_error,
        model_is_better,
        model_is_equal
    ) in zip(
        test_row_indices,
        actual_values,
        predicted_values,
        median_predictions,
        absolute_errors,
        median_absolute_errors,
        better_than_median,
        equal_to_median
    ):
        all_estimates.append({
            "Test-rad": int(row_index),
            "Variabel": column_name,
            "Originalvärde": actual_value,
            "Autoencoder": predicted_value,
            "Median": median_prediction,
            "Autoencoder-fel": absolute_error,
            "Median-fel": median_absolute_error,
            "Autoencoder bättre": model_is_better,
            "Samma fel": model_is_equal
        })


evaluation_df = pd.DataFrame(evaluation_rows)
all_estimates_df = pd.DataFrame(all_estimates)


# ============================================================
# Global utvärdering i standardiserad skala
# ============================================================
# Variablerna har olika enheter och skalor:
#
# MedInc     -> inkomst
# HouseAge   -> år
# Population -> antal personer
# Latitude   -> koordinat
#
# Därför bör deras råa fel inte slås ihop direkt.
#
# Vi beräknar i stället de globala måtten i standardiserad skala.
# Då är samtliga variabler jämförbara.
# ============================================================

actual_hidden_scaled = test_original_scaled[
    test_missing_mask
]

predicted_hidden_scaled = reconstructed_test_scaled[
    test_missing_mask
]

# Median-baseline i standardiserad skala.
median_matrix_scaled = np.broadcast_to(
    median_scaled,
    test_original_scaled.shape
)

median_hidden_scaled = median_matrix_scaled[
    test_missing_mask
]

global_model_mae = mean_absolute_error(
    actual_hidden_scaled,
    predicted_hidden_scaled
)

global_median_mae = mean_absolute_error(
    actual_hidden_scaled,
    median_hidden_scaled
)

global_model_rmse = np.sqrt(
    mean_squared_error(
        actual_hidden_scaled,
        predicted_hidden_scaled
    )
)

global_median_rmse = np.sqrt(
    mean_squared_error(
        actual_hidden_scaled,
        median_hidden_scaled
    )
)

global_r2 = r2_score(
    actual_hidden_scaled,
    predicted_hidden_scaled
)

global_mae_improvement = (
    (global_median_mae - global_model_mae)
    / (global_median_mae + 1e-8)
    * 100
)

global_absolute_errors = np.abs(
    actual_hidden_scaled - predicted_hidden_scaled
)

global_median_absolute_errors = np.abs(
    actual_hidden_scaled - median_hidden_scaled
)

global_better_than_median = np.mean(
    global_absolute_errors
    < global_median_absolute_errors
)

# Eftersom värdena är standardiserade motsvarar:
#
# 0.10 = 10 % av en standardavvikelse
# 0.25 = 25 % av en standardavvikelse
global_within_10_percent_std = np.mean(
    global_absolute_errors <= 0.10
)

global_within_25_percent_std = np.mean(
    global_absolute_errors <= 0.25
)


# ============================================================
# Skriv ut resultat per variabel
# ============================================================

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)

print("\n")
print("=" * 150)
print("MODELLENS RESULTAT PER VARIABEL")
print("=" * 150)

print(
    evaluation_df.round({
        "Autoencoder MAE": 4,
        "Median MAE": 4,
        "Förbättring mot median (%)": 2,
        "RMSE": 4,
        "R2": 4,
        "Bättre än median (%)": 2,
        "Inom 10 % av std (%)": 2,
        "Inom 25 % av std (%)": 2
    }).to_string(index=False)
)


# ============================================================
# Skriv ut global sammanfattning
# ============================================================

print("\n")
print("=" * 80)
print("TOTAL PRESTANDA PÅ SAMTLIGA DOLDA VÄRDEN")
print("=" * 80)

print(
    f"Totalt antal dolda och skattade värden: "
    f"{len(actual_hidden_scaled)}"
)

print("\nResultat i standardiserad skala:")

print(
    f"Autoencoder MAE:                 "
    f"{global_model_mae:.4f}"
)

print(
    f"Median MAE:                      "
    f"{global_median_mae:.4f}"
)

print(
    f"Autoencoder RMSE:                "
    f"{global_model_rmse:.4f}"
)

print(
    f"Median RMSE:                     "
    f"{global_median_rmse:.4f}"
)

print(
    f"Autoencoder R²:                  "
    f"{global_r2:.4f}"
)

print(
    f"MAE-förbättring mot median:      "
    f"{global_mae_improvement:.2f}%"
)

print(
    f"Autoencoder bättre än median:    "
    f"{global_better_than_median:.2%}"
)

print(
    f"Fel inom 10 % av std:            "
    f"{global_within_10_percent_std:.2%}"
)

print(
    f"Fel inom 25 % av std:            "
    f"{global_within_25_percent_std:.2%}"
)


# ============================================================
# Automatisk övergripande bedömning
# ============================================================

print("\n")
print("=" * 80)
print("AUTOMATISK BEDÖMNING")
print("=" * 80)

if global_mae_improvement > 20:
    print(
        "Modellen presterar tydligt bättre än medianimputering."
    )

elif global_mae_improvement > 5:
    print(
        "Modellen presterar bättre än medianimputering, "
        "men förbättringen är måttlig."
    )

elif global_mae_improvement >= 0:
    print(
        "Modellen är endast marginellt bättre än "
        "medianimputering."
    )

else:
    print(
        "Modellen presterar sämre än medianimputering."
    )


if global_r2 >= 0.75:
    print(
        "R² är högt: modellen återskapar en stor del av "
        "variationen i de dolda värdena."
    )

elif global_r2 >= 0.50:
    print(
        "R² är måttligt till bra: modellen återskapar en "
        "betydande del av variationen."
    )

elif global_r2 >= 0.25:
    print(
        "R² är relativt lågt: modellen fångar en del av "
        "variationen, men rekonstruktionerna är osäkra."
    )

elif global_r2 >= 0:
    print(
        "R² är lågt: modellen återskapar endast en liten del "
        "av variationen."
    )

else:
    print(
        "R² är negativt: modellen är sämre än en enkel "
        "konstant baseline sett till variationen."
    )


# ============================================================
# Skriv ut bästa och sämsta variabler
# ============================================================

best_feature = evaluation_df.loc[
    evaluation_df[
        "Förbättring mot median (%)"
    ].idxmax()
]

worst_feature = evaluation_df.loc[
    evaluation_df[
        "Förbättring mot median (%)"
    ].idxmin()
]

print("\n")
print("=" * 80)
print("BÄSTA OCH SÄMSTA VARIABEL")
print("=" * 80)

print(
    f"Bäst återskapad variabel: "
    f"{best_feature['Variabel']}"
)

print(
    f"Förbättring mot median: "
    f"{best_feature['Förbättring mot median (%)']:.2f}%"
)

print(
    f"R²: {best_feature['R2']:.4f}"
)

print()

print(
    f"Sämst återskapad variabel: "
    f"{worst_feature['Variabel']}"
)

print(
    f"Förbättring mot median: "
    f"{worst_feature['Förbättring mot median (%)']:.2f}%"
)

print(
    f"R²: {worst_feature['R2']:.4f}"
)


# ============================================================
# Skriv ut varje enskild skattning
# ============================================================
# Detta kan ge flera tusen rader i terminalen.
#
# Sätt SHOW_ALL_ESTIMATES = False om du bara vill se ett urval.
# ============================================================

SHOW_ALL_ESTIMATES = True

print("\n")
print("=" * 150)
print("ENSKILDA SKATTNINGAR")
print("=" * 150)

if SHOW_ALL_ESTIMATES:
    print(
        all_estimates_df
        .sort_values(["Variabel", "Test-rad"])
        .round(4)
        .to_string(index=False)
    )

else:
    # Visa de första 20 skattningarna per variabel.
    estimates_to_show = (
        all_estimates_df
        .sort_values(["Variabel", "Test-rad"])
        .groupby("Variabel")
        .head(20)
    )

    print(
        estimates_to_show
        .round(4)
        .to_string(index=False)
    )


# ============================================================
# Skriv ut de största rekonstruktionsfelen
# ============================================================

print("\n")
print("=" * 150)
print("DE 20 STÖRSTA REKONSTRUKTIONSFELEN")
print("=" * 150)

print(
    all_estimates_df
    .sort_values(
        "Autoencoder-fel",
        ascending=False
    )
    .head(20)
    .round(4)
    .to_string(index=False)
)



MODELLENS RESULTAT PER VARIABEL
  Variabel  Antal skattningar  Autoencoder MAE  Median MAE  Förbättring mot median (%)     RMSE      R2  Bättre än median (%)  Inom 10 % av std (%)  Inom 25 % av std (%)
    MedInc                684           0.8411      1.4500                       41.99   1.1462  0.6853                 69.01                 16.52                 38.60
  HouseAge                661           8.2412     10.5431                       21.83  10.6528  0.2889                 67.32                  9.53                 25.72
  AveRooms                678           0.6393      1.0741                       40.48   0.9250  0.7844                 68.58                 25.81                 61.65
 AveBedrms                707           0.0764      0.1017                       24.82   0.1359  0.8230                 53.18                 45.83                 82.60
Population                687         589.7450    628.0160                        6.09 884.7268  0.1318             